# Goodreads Review Popularity — Memory-Safe Preprocessing

This notebook reimplements the preprocessing workflow from the uploaded `data_prep.py` using **pandas, NumPy, SQLite, and FastText**. It does not use DuckDB and does not load the full review dataset into RAM.

### Workflow
1. Stream review JSONL in chunks and discard zero-engagement reviews.
2. Build book-level and user-level counts in disk-backed SQLite tables.
3. Stream book metadata and retain books with `text_reviews_count >= 10`.
4. Keep books with at least 60 combined votes/comments across retained reviews.
5. Calculate `like_share` and the binary `popular` target.
6. Detect English reviews in small batches using FastText.
7. Add `days_since_review` and `user_reviews`, then append results to CSV.

**Threshold choice:** uploaded repository code uses `0.02`; the paper states `0.025`. The configuration defaults to repository mode. Change `REPLICATION_MODE` to `paper` to use `0.025`. This is a memory-safe reimplementation of the script's workflow, not a byte-for-byte copy.

## 1. Install dependencies

Run this in Google Colab. The language-identification model is downloaded separately later.

In [ ]:
%pip -q install fasttext-wheel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for fasttext-wheel (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for fasttext-wheel
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (fasttext-wheel)


## 2. Configure paths and settings

Update the paths to wherever you uploaded/mounted the raw files. Supported formats are `.json` and `.json.gz` JSON Lines files.

In [ ]:
from pathlib import Path
import shutil
import subprocess

DATA_DIR = Path("/content/goodreads_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Official UCSD Goodreads dataset download URLs
datasets = {
    "goodreads_reviews_dedup.json.gz":
        "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_reviews_dedup.json.gz",
    "goodreads_books.json.gz":
        "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_books.json.gz",
}

print("Available disk space before download:")
print(shutil.disk_usage("/content"))

for filename, url in datasets.items():
    destination = DATA_DIR / filename

    if destination.exists() and destination.stat().st_size > 0:
        print(f"Already exists, skipping: {filename}")
        continue

    print(f"\nDownloading {filename}...")
    result = subprocess.run([
        "wget", "-c", "--show-progress",
        "-O", str(destination), url
    ])

    if result.returncode != 0:
        print(f"Download failed: {filename}")
        print("Rerun this cell to attempt resuming the download.")
        break

    print(f"Downloaded: {destination}")

print("\nDataset directory:", DATA_DIR)
print("Files:")
for file in DATA_DIR.glob("*"):
    print(f"{file.name}: {file.stat().st_size / (1024**3):.2f} GB")

Available disk space before download:
usage(total=115658190848, used=22007492608, free=93633921024)

Downloaded: /content/goodreads_data/goodreads_reviews_dedup.json.gz

Downloaded: /content/goodreads_data/goodreads_books.json.gz

Dataset directory: /content/goodreads_data
Files:
goodreads_books.json.gz: 1.94 GB
goodreads_reviews_dedup.json.gz: 4.98 GB


In [ ]:
REVIEWS_PATH = Path(
    "/content/goodreads_data/goodreads_reviews_dedup.json.gz"
)

BOOKS_PATH = Path(
    "/content/goodreads_data/goodreads_books.json.gz"
)

## 3. Check input files

In [ ]:
for p in (REVIEWS_PATH, BOOKS_PATH):
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}. Upload/mount it or update the path.")
    print(f"{p.name}: {p.stat().st_size / (1024**3):.2f} GiB")

goodreads_reviews_dedup.json.gz: 4.98 GiB
goodreads_books.json.gz: 1.94 GiB


In [ ]:
from pathlib import Path
import sqlite3, time, urllib.request
import numpy as np
import pandas as pd

REVIEWS_PATH = Path("/content/goodreads_reviews_dedup.json.gz")
BOOKS_PATH = Path("/content/goodreads_books.json.gz")  # or .json
WORK_DIR = Path("/content/goodreads_preprocessing")
WORK_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = WORK_DIR / "goodreads_stats.sqlite"
OUTPUT_CSV = WORK_DIR / "filtered_reviews.csv"
FASTTEXT_MODEL_PATH = WORK_DIR / "lid.176.bin"

# Lower these if RAM is tight.
REVIEW_CHUNK_SIZE = 5_000
BOOK_CHUNK_SIZE = 10_000
LANGUAGE_BATCH_SIZE = 500

REPLICATION_MODE = "repository"  # set to "paper" for paper threshold 0.025
POPULAR_THRESHOLD = 0.02 if REPLICATION_MODE == "repository" else 0.025
ENGLISH_PROBABILITY_THRESHOLD = 0.90

print("Review file:", REVIEWS_PATH)
print("Book file:", BOOKS_PATH)
print("Mode / popularity threshold:", REPLICATION_MODE, POPULAR_THRESHOLD)
print("Output:", OUTPUT_CSV)

Review file: /content/goodreads_reviews_dedup.json.gz
Book file: /content/goodreads_books.json.gz
Mode / popularity threshold: repository 0.02
Output: /content/goodreads_preprocessing/filtered_reviews.csv


## 4. Helpers for chunked reading and normalization

In [ ]:

def iter_jsonl_chunks(path, chunksize):
    """Read JSON Lines in chunks without automatically parsing dates."""
    compression = "gzip" if str(path).endswith(".gz") else None

    yield from pd.read_json(
        path,
        lines=True,
        chunksize=chunksize,
        compression=compression,
        convert_dates=False
    )



def normalize_review_chunk(df: pd.DataFrame) -> pd.DataFrame:
    required = ["user_id", "book_id", "rating", "review_text", "date_added", "n_votes", "n_comments"]
    for col in required:
        if col not in df.columns:
            df[col] = np.nan
    df["user_id"] = df["user_id"].astype("string")
    df["book_id"] = df["book_id"].astype("string")
    df["n_votes"] = pd.to_numeric(df["n_votes"], errors="coerce").fillna(0)
    df["n_comments"] = pd.to_numeric(df["n_comments"], errors="coerce").fillna(0)
    df["review_likes"] = df["n_votes"] + df["n_comments"]
    df["review_text"] = df["review_text"].fillna("").astype(str)
    df["date_added"] = df["date_added"].astype("string")
    return df


def parse_review_dates(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, format="%a %b %d %H:%M:%S %z %Y", errors="coerce", utc=True)

print("Helpers ready")

Helpers ready


## 5. Pass one: build book and user statistics in SQLite

The original script first removes reviews where `n_votes + n_comments == 0`. These counts are calculated only from the retained reviews. SQLite stores aggregate counts on disk instead of keeping all review records or a huge Python dictionary in memory.

In [ ]:
import os

# Ensure paths point to the correct subdirectory if they don't exist in /content/
if not REVIEWS_PATH.exists() and Path("/content/goodreads_data/goodreads_reviews_dedup.json.gz").exists():
    REVIEWS_PATH = Path("/content/goodreads_data/goodreads_reviews_dedup.json.gz")
if not BOOKS_PATH.exists() and Path("/content/goodreads_data/goodreads_books.json.gz").exists():
    BOOKS_PATH = Path("/content/goodreads_data/goodreads_books.json.gz")

if DB_PATH.exists():
    DB_PATH.unlink()  # prevents double-counting if the notebook is rerun
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA synchronous=NORMAL")
conn.execute("PRAGMA temp_store=FILE")
conn.execute("CREATE TABLE book_stats (book_id TEXT PRIMARY KEY, review_count INTEGER NOT NULL, total_engagement REAL NOT NULL)")
conn.execute("CREATE TABLE user_stats (user_id TEXT PRIMARY KEY, user_reviews INTEGER NOT NULL)")
conn.commit()

start_time = time.time()
total_reviews_seen = 0
engaged_reviews_seen = 0

for chunk_no, raw in enumerate(iter_jsonl_chunks(REVIEWS_PATH, REVIEW_CHUNK_SIZE), start=1):
    total_reviews_seen += len(raw)
    chunk = normalize_review_chunk(raw)
    chunk = chunk.loc[(chunk["review_likes"] > 0) & chunk["book_id"].notna() & chunk["user_id"].notna()].copy()
    engaged_reviews_seen += len(chunk)
    if chunk.empty:
        continue

    book_agg = chunk.groupby("book_id").agg(review_count=("book_id", "size"), total_engagement=("review_likes", "sum")).reset_index()
    conn.executemany(
        """INSERT INTO book_stats VALUES (?, ?, ?)
        ON CONFLICT(book_id) DO UPDATE SET
        review_count=review_count+excluded.review_count,
        total_engagement=total_engagement+excluded.total_engagement""",
        [(str(r.book_id), int(r.review_count), float(r.total_engagement)) for r in book_agg.itertuples(index=False)]
    )
    user_agg = chunk.groupby("user_id").size().rename("user_reviews").reset_index()
    conn.executemany(
        """INSERT INTO user_stats VALUES (?, ?)
        ON CONFLICT(user_id) DO UPDATE SET user_reviews=user_reviews+excluded.user_reviews""",
        [(str(r.user_id), int(r.user_reviews)) for r in user_agg.itertuples(index=False)]
    )
    conn.commit()
    if chunk_no % 200 == 0:
        print(f"Chunks {chunk_no:,} | scanned {total_reviews_seen:,} | engaged {engaged_reviews_seen:,}")

print(f"Pass 1 took {(time.time()-start_time)/60:.1f} minutes")
print("Reviews scanned:", f"{total_reviews_seen:,}")
print("Engaged reviews:", f"{engaged_reviews_seen:,}")
print("Book IDs in stats:", f"{conn.execute('SELECT COUNT(*) FROM book_stats').fetchone()[0]:,}")
print("User IDs in stats:", f"{conn.execute('SELECT COUNT(*) FROM user_stats').fetchone()[0]:,}")

Chunks 200 | scanned 1,000,000 | engaged 307,755
Chunks 400 | scanned 2,000,000 | engaged 597,412
Chunks 600 | scanned 3,000,000 | engaged 896,287
Chunks 800 | scanned 4,000,000 | engaged 1,197,219
Chunks 1,000 | scanned 5,000,000 | engaged 1,503,627
Chunks 1,200 | scanned 6,000,000 | engaged 1,809,362
Chunks 1,400 | scanned 7,000,000 | engaged 2,123,920
Chunks 1,600 | scanned 8,000,000 | engaged 2,441,187
Chunks 1,800 | scanned 9,000,000 | engaged 2,759,201
Chunks 2,000 | scanned 10,000,000 | engaged 3,076,781
Chunks 2,200 | scanned 11,000,000 | engaged 3,379,360
Chunks 2,400 | scanned 12,000,000 | engaged 3,681,974
Chunks 2,600 | scanned 13,000,000 | engaged 3,983,101
Chunks 2,800 | scanned 14,000,000 | engaged 4,273,417
Chunks 3,000 | scanned 15,000,000 | engaged 4,556,312
Pass 1 took 12.0 minutes
Reviews scanned: 15,739,967
Engaged reviews: 4,706,961
Book IDs in stats: 1,042,951
User IDs in stats: 267,674


## 6. Stream book metadata and keep books with 10+ text reviews

The original script filters book metadata using `text_reviews_count >= 10`, retaining only fields needed later.

In [ ]:

import gzip
import json
import time
import pandas as pd
import numpy as np

# Reset the table so rerunning this cell does not duplicate data.
conn.execute("DROP TABLE IF EXISTS eligible_books")
conn.execute("""
CREATE TABLE eligible_books (
    book_id TEXT PRIMARY KEY,
    text_reviews_count INTEGER,
    ratings_count REAL,
    average_rating REAL
)
""")
conn.commit()

def open_jsonl(path):
    """Open compressed or uncompressed JSON Lines files."""
    if str(path).endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "rt", encoding="utf-8")

def safe_number(value):
    """Convert a value to a number; return None if invalid."""
    try:
        if value is None:
            return None
        number = float(value)
        return number if np.isfinite(number) else None
    except (ValueError, TypeError):
        return None

book_rows_seen = 0
book_rows_10plus = 0
batch = []
BATCH_SIZE = 5000
start_time = time.time()

with open_jsonl(BOOKS_PATH) as f:
    for line in f:
        if not line.strip():
            continue

        book_rows_seen += 1

        try:
            entry = json.loads(line)

            book_id = entry.get("book_id")
            review_count = safe_number(
                entry.get("text_reviews_count")
            )

            if (
                book_id is not None
                and review_count is not None
                and review_count >= 10
            ):
                batch.append((
                    str(book_id),
                    int(review_count),
                    safe_number(entry.get("ratings_count")),
                    safe_number(entry.get("average_rating"))
                ))

                book_rows_10plus += 1

        except (json.JSONDecodeError, TypeError, ValueError):
            continue

        # Insert small batches into SQLite instead of keeping
        # the entire book dataset in RAM.
        if len(batch) >= BATCH_SIZE:
            conn.executemany("""
                INSERT OR REPLACE INTO eligible_books
                (book_id, text_reviews_count, ratings_count, average_rating)
                VALUES (?, ?, ?, ?)
            """, batch)

            conn.commit()
            batch.clear()

        if book_rows_seen % 100000 == 0:
            print(
                f"Books scanned: {book_rows_seen:,} | "
                f"Books with 10+ reviews: {book_rows_10plus:,} | "
                f"Elapsed: {(time.time()-start_time)/60:.1f} min"
            )

# Insert the final partial batch.
if batch:
    conn.executemany("""
        INSERT OR REPLACE INTO eligible_books
        (book_id, text_reviews_count, ratings_count, average_rating)
        VALUES (?, ?, ?, ?)
    """, batch)
    conn.commit()

print("\nBook metadata processing complete.")
print(f"Total records scanned: {book_rows_seen:,}")
print(f"Records with 10+ text reviews: {book_rows_10plus:,}")
print(
    "Unique eligible book IDs:",
    conn.execute(
        "SELECT COUNT(*) FROM eligible_books"
    ).fetchone()[0]
)
print(f"Time taken: {(time.time()-start_time)/60:.1f} minutes")


Books scanned: 100,000 | Books with 10+ reviews: 29,054 | Elapsed: 0.2 min
Books scanned: 200,000 | Books with 10+ reviews: 57,628 | Elapsed: 0.4 min
Books scanned: 300,000 | Books with 10+ reviews: 86,088 | Elapsed: 0.5 min
Books scanned: 400,000 | Books with 10+ reviews: 114,329 | Elapsed: 0.7 min
Books scanned: 500,000 | Books with 10+ reviews: 142,477 | Elapsed: 0.8 min
Books scanned: 600,000 | Books with 10+ reviews: 170,275 | Elapsed: 1.0 min
Books scanned: 700,000 | Books with 10+ reviews: 198,119 | Elapsed: 1.1 min
Books scanned: 800,000 | Books with 10+ reviews: 226,077 | Elapsed: 1.3 min
Books scanned: 900,000 | Books with 10+ reviews: 253,670 | Elapsed: 1.4 min
Books scanned: 1,000,000 | Books with 10+ reviews: 281,324 | Elapsed: 1.6 min
Books scanned: 1,100,000 | Books with 10+ reviews: 308,944 | Elapsed: 1.8 min
Books scanned: 1,200,000 | Books with 10+ reviews: 336,651 | Elapsed: 1.9 min
Books scanned: 1,300,000 | Books with 10+ reviews: 365,714 | Elapsed: 2.1 min
Books s

## 7. Qualify books using total engagement >= 60

The original script joins reviews to books with at least 10 text reviews, then keeps books with at least 60 combined votes/comments across retained reviews.

In [ ]:
conn.execute("DROP TABLE IF EXISTS qualifying_books")
conn.execute("""CREATE TABLE qualifying_books AS
SELECT e.book_id, e.text_reviews_count, e.ratings_count, e.average_rating,
       s.review_count, s.total_engagement
FROM eligible_books e JOIN book_stats s ON e.book_id=s.book_id
WHERE s.total_engagement >= 60""")
conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_qualifying_book_id ON qualifying_books(book_id)")
conn.commit()
qualifying_books = {r[0] for r in conn.execute("SELECT book_id FROM qualifying_books").fetchall()}
print("Qualifying books:", f"{len(qualifying_books):,}")

Qualifying books: 58,309


## 8. Download and load FastText language model

This is the same language-identification model family used in the original script. The model is saved to disk so it does not need to be downloaded on every run.

In [ ]:
import urllib.request
import sys
import subprocess

# 1. Install fasttext library
try:
    import fasttext
except ImportError:
    print("Installing fasttext...")
    # Using --no-build-isolation can sometimes bypass wheel build issues, or we use standard pip
    subprocess.run([sys.executable, "-m", "pip", "install", "fasttext"])
    import fasttext

# 2. Define the download URL for the language identification model
FASTTEXT_MODEL_URL = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"

# 3. Download the model if it doesn't already exist
if not FASTTEXT_MODEL_PATH.exists():
    print(f"Downloading FastText language detection model from: {FASTTEXT_MODEL_URL}")
    urllib.request.urlretrieve(FASTTEXT_MODEL_URL, FASTTEXT_MODEL_PATH)
    print(f"Saved model to {FASTTEXT_MODEL_PATH}")
else:
    print(f"FastText model already exists at {FASTTEXT_MODEL_PATH}")

# 4. Load the model to verify it works
language_model = fasttext.load_model(str(FASTTEXT_MODEL_PATH))
print("FastText model loaded successfully!")

Installing fasttext...
FastText model already exists at /content/goodreads_preprocessing/lid.176.bin
FastText model loaded successfully!


## 9. Find the latest review date for qualifying books

The original code calculates `days_since_review` relative to the maximum parsed `date_added` in the filtered dataset. This extra pass avoids storing all qualifying dates in RAM.

In [ ]:

import gzip
import json
import time
import pandas as pd
from datetime import datetime, timezone

def open_jsonl(path):
    if str(path).endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "rt", encoding="utf-8")

max_review_date = None
total_scanned = 0
qualifying_date_rows = 0
start_time = time.time()

with open_jsonl(REVIEWS_PATH) as f:
    for line in f:
        if not line.strip():
            continue

        total_scanned += 1

        try:
            entry = json.loads(line)

            book_id = str(entry.get("book_id", ""))
            if book_id not in qualifying_books:
                continue

            votes = pd.to_numeric(
                entry.get("n_votes", 0), errors="coerce"
            )
            comments = pd.to_numeric(
                entry.get("n_comments", 0), errors="coerce"
            )

            engagement = (0 if pd.isna(votes) else votes) + (
                0 if pd.isna(comments) else comments
            )

            if engagement <= 0:
                continue

            date_string = entry.get("date_added")
            if not date_string:
                continue

            try:
                review_date = datetime.strptime(
                    date_string, "%a %b %d %H:%M:%S %z %Y"
                ).astimezone(timezone.utc)
            except (ValueError, TypeError):
                continue

            qualifying_date_rows += 1

            if max_review_date is None or review_date > max_review_date:
                max_review_date = review_date

        except (json.JSONDecodeError, TypeError, ValueError):
            continue

        if total_scanned % 100_000 == 0:
            print(
                f"Scanned {total_scanned:,} reviews | "
                f"Eligible dated reviews: {qualifying_date_rows:,} | "
                f"Elapsed: {(time.time() - start_time) / 60:.1f} min"
            )

if max_review_date is None:
    raise ValueError(
        "No valid dates found for qualifying books. "
        "Check the date_added format and qualifying_books."
    )

max_review_date = pd.Timestamp(max_review_date)

print("\nDate scan completed.")
print("Total reviews scanned:", f"{total_scanned:,}")
print("Eligible reviews with valid dates:", f"{qualifying_date_rows:,}")
print("Maximum date:", max_review_date)
print("Time taken:", f"{(time.time() - start_time) / 60:.1f} minutes")


Scanned 200,000 reviews | Eligible dated reviews: 31,873 | Elapsed: 0.1 min
Scanned 600,000 reviews | Eligible dated reviews: 94,779 | Elapsed: 0.3 min
Scanned 1,800,000 reviews | Eligible dated reviews: 267,610 | Elapsed: 0.9 min
Scanned 1,900,000 reviews | Eligible dated reviews: 284,143 | Elapsed: 1.0 min
Scanned 2,000,000 reviews | Eligible dated reviews: 300,708 | Elapsed: 1.1 min
Scanned 2,200,000 reviews | Eligible dated reviews: 330,895 | Elapsed: 1.2 min
Scanned 2,500,000 reviews | Eligible dated reviews: 376,236 | Elapsed: 1.3 min
Scanned 3,000,000 reviews | Eligible dated reviews: 451,033 | Elapsed: 1.6 min
Scanned 3,100,000 reviews | Eligible dated reviews: 466,905 | Elapsed: 1.6 min
Scanned 4,100,000 reviews | Eligible dated reviews: 615,867 | Elapsed: 2.1 min
Scanned 6,000,000 reviews | Eligible dated reviews: 905,355 | Elapsed: 3.1 min
Scanned 6,300,000 reviews | Eligible dated reviews: 952,979 | Elapsed: 3.2 min
Scanned 6,800,000 reviews | Eligible dated reviews: 1,032,

## 10. Final pass: filter English reviews and write CSV incrementally

The output schema matches the uploaded repository script. FastText is called on small batches. Only one chunk plus its language predictions is held in memory at a time.

In [ ]:

import gzip
import json
import time
import sqlite3
import numpy as np
import pandas as pd
from pathlib import Path

# Assumes these already exist from earlier cells:
# REVIEWS_PATH, OUTPUT_CSV, DB_PATH, qualifying_books,
# max_review_date, language_model, POPULAR_THRESHOLD

CHECKPOINT_PATH = OUTPUT_CSV.with_suffix(".checkpoint.json")
BATCH_SIZE = 500
LOG_EVERY = 50_000

OUTPUT_COLUMNS = [
    "book_id", "user_reviews", "user_rating", "avg_rating",
    "ratings_count", "review_text", "days_since_review",
    "review_likes", "like_share", "popular"
]

def open_review_file(path):
    if str(path).endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "rt", encoding="utf-8")

def as_number(value, default=0):
    try:
        result = float(value)
        return result if np.isfinite(result) else default
    except (TypeError, ValueError):
        return default

def parse_date(value):
    if not value:
        return pd.NaT
    return pd.to_datetime(
        value,
        format="%a %b %d %H:%M:%S %z %Y",
        errors="coerce",
        utc=True
    )

# Read saved progress, if available.
if CHECKPOINT_PATH.exists() and OUTPUT_CSV.exists():
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        checkpoint = json.load(f)
    start_line = int(checkpoint["last_line"]) + 1
    print(f"Resuming from review line {start_line:,}")
else:
    start_line = 1
    if OUTPUT_CSV.exists():
        OUTPUT_CSV.unlink()

    # A new CSV requires a header.
    pd.DataFrame(columns=OUTPUT_COLUMNS).to_csv(
        OUTPUT_CSV, index=False
    )
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump({"last_line": 0}, f)

conn = sqlite3.connect(DB_PATH)

def lookup_book_data(book_ids):
    result = {}
    unique_ids = list(set(book_ids))

    for i in range(0, len(unique_ids), 500):
        ids = unique_ids[i:i + 500]
        if not ids:
            continue
        placeholders = ",".join("?" for _ in ids)
        query = f"""
            SELECT book_id, text_reviews_count, ratings_count,
                   average_rating, total_engagement
            FROM qualifying_books
            WHERE book_id IN ({placeholders})
        """
        for row in conn.execute(query, ids):
            result[row[0]] = row[1:]
    return result

def lookup_user_counts(user_ids):
    result = {}
    unique_ids = list(set(user_ids))

    for i in range(0, len(unique_ids), 500):
        ids = unique_ids[i:i + 500]
        if not ids:
            continue
        placeholders = ",".join("?" for _ in ids)
        query = f"""
            SELECT user_id, user_reviews
            FROM user_stats
            WHERE user_id IN ({placeholders})
        """
        for row in conn.execute(query, ids):
            result[row[0]] = row[1]
    return result

# Open output in append mode and process the archive sequentially.
buffer = []
total_lines = 0
eligible_seen = 0
written_this_run = 0
started = time.time()
last_saved_line = start_line - 1

with open_review_file(REVIEWS_PATH) as f:
    for line_no, line in enumerate(f, start=1):
        total_lines = line_no

        if line_no < start_line:
            continue

        if not line.strip():
            last_saved_line = line_no
            continue

        try:
            entry = json.loads(line)
        except json.JSONDecodeError:
            last_saved_line = line_no
            continue

        book_id = str(entry.get("book_id", ""))
        if book_id not in qualifying_books:
            last_saved_line = line_no
            continue

        engagement = (
            as_number(entry.get("n_votes")) +
            as_number(entry.get("n_comments"))
        )
        if engagement <= 0:
            last_saved_line = line_no
            continue

        date_value = entry.get("date_added")
        review_date = parse_date(date_value)
        if pd.isna(review_date):
            last_saved_line = line_no
            continue

        text = str(entry.get("review_text") or "").replace("\n", " ")
        buffer.append({
            "book_id": book_id,
            "user_id": str(entry.get("user_id", "")),
            "user_rating": entry.get("rating"),
            "review_text": text,
            "review_date": review_date,
            "review_likes": engagement
        })
        eligible_seen += 1
        last_saved_line = line_no

        if len(buffer) >= BATCH_SIZE:
            batch_df = pd.DataFrame(buffer)
            buffer = []

            book_data = lookup_book_data(batch_df["book_id"].tolist())
            user_data = lookup_user_counts(batch_df["user_id"].tolist())

            batch_df["total_engagement"] = batch_df["book_id"].map(
                lambda x: book_data.get(x, (None, None, None, None))[3]
            )
            batch_df["avg_rating"] = batch_df["book_id"].map(
                lambda x: book_data.get(x, (None, None, None, None))[2]
            )
            batch_df["ratings_count"] = batch_df["book_id"].map(
                lambda x: book_data.get(x, (None, None, None, None))[1]
            )
            batch_df["user_reviews"] = batch_df["user_id"].map(user_data)

            batch_df["like_share"] = (
                batch_df["review_likes"] /
                pd.to_numeric(batch_df["total_engagement"], errors="coerce")
            )
            batch_df["popular"] = np.where(
                batch_df["like_share"] > POPULAR_THRESHOLD, 1, 0
            )
            batch_df["days_since_review"] = (
                pd.Timestamp(max_review_date) - batch_df["review_date"]
            ).dt.days

            texts = [
                t if t.strip() else "empty review"
                for t in batch_df["review_text"].tolist()
            ]
            keep = []

            for i in range(0, len(texts), 100):
                labels, probs = language_model.predict(texts[i:i + 100], k=1)
                for j, (label_list, prob_list) in enumerate(zip(labels, probs)):
                    if (
                        label_list
                        and label_list[0] == "__label__en"
                        and float(prob_list[0]) > 0.90
                    ):
                        keep.append(i + j)

            if keep:
                out = batch_df.iloc[keep][[
                    "book_id", "user_reviews", "user_rating", "avg_rating",
                    "ratings_count", "review_text", "days_since_review",
                    "review_likes", "like_share", "popular"
                ]].copy()

                out.to_csv(
                    OUTPUT_CSV, mode="a",
                    header=False, index=False
                )
                written_this_run += len(out)

            # Save checkpoint only after all rows through this batch were written.
            with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
                json.dump({"last_line": last_saved_line}, f)

        if line_no % LOG_EVERY == 0:
            print(
                f"Line {line_no:,} | eligible buffered/processed: "
                f"{eligible_seen:,} | output rows this run: "
                f"{written_this_run:,} | elapsed: "
                f"{(time.time()-started)/60:.1f} min"
            )

# Process any remaining records in the buffer using the same logic
# by rerunning with a larger batch is not required; use the block below.
if buffer:
    batch_df = pd.DataFrame(buffer)
    book_data = lookup_book_data(batch_df["book_id"].tolist())
    user_data = lookup_user_counts(batch_df["user_id"].tolist())

    batch_df["total_engagement"] = batch_df["book_id"].map(
        lambda x: book_data.get(x, (None, None, None, None))[3]
    )
    batch_df["avg_rating"] = batch_df["book_id"].map(
        lambda x: book_data.get(x, (None, None, None, None))[2]
    )
    batch_df["ratings_count"] = batch_df["book_id"].map(
        lambda x: book_data.get(x, (None, None, None, None))[1]
    )
    batch_df["user_reviews"] = batch_df["user_id"].map(user_data)
    batch_df["like_share"] = (
        batch_df["review_likes"] /
        pd.to_numeric(batch_df["total_engagement"], errors="coerce")
    )
    batch_df["popular"] = np.where(
        batch_df["like_share"] > POPULAR_THRESHOLD, 1, 0
    )
    batch_df["days_since_review"] = (
        pd.Timestamp(max_review_date) - batch_df["review_date"]
    ).dt.days

    texts = [
        t if t.strip() else "empty review"
        for t in batch_df["review_text"].tolist()
    ]
    keep = []
    for i in range(0, len(texts), 100):
        labels, probs = language_model.predict(texts[i:i + 100], k=1)
        for j, (label_list, prob_list) in enumerate(zip(labels, probs)):
            if (
                label_list
                and label_list[0] == "__label__en"
                and float(prob_list[0]) > 0.90
            ):
                keep.append(i + j)

    if keep:
        out = batch_df.iloc[keep][OUTPUT_COLUMNS].copy()
        out.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)
        written_this_run += len(out)

# Mark completion only after the final buffered rows have been written.
with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
    json.dump({"last_line": total_lines, "complete": True}, f)

conn.close()

print("\nProcessing complete.")
print("Output CSV:", OUTPUT_CSV)
print("Rows written in this run:", f"{written_this_run:,}")
print("Elapsed minutes:", round((time.time()-started)/60, 2))


Line 200,000 | eligible buffered/processed: 31,873 | output rows this run: 27,953 | elapsed: 0.4 min
Line 600,000 | eligible buffered/processed: 94,779 | output rows this run: 81,858 | elapsed: 1.0 min
Line 750,000 | eligible buffered/processed: 116,654 | output rows this run: 100,663 | elapsed: 1.3 min
Line 1,050,000 | eligible buffered/processed: 161,732 | output rows this run: 138,964 | elapsed: 1.7 min
Line 1,450,000 | eligible buffered/processed: 217,161 | output rows this run: 185,568 | elapsed: 2.4 min
Line 1,800,000 | eligible buffered/processed: 267,610 | output rows this run: 227,804 | elapsed: 3.0 min
Line 1,900,000 | eligible buffered/processed: 284,143 | output rows this run: 241,941 | elapsed: 3.1 min
Line 1,950,000 | eligible buffered/processed: 291,917 | output rows this run: 248,479 | elapsed: 3.2 min
Line 2,000,000 | eligible buffered/processed: 300,708 | output rows this run: 256,619 | elapsed: 3.3 min
Line 2,050,000 | eligible buffered/processed: 307,749 | output ro

## 11. Validate output without loading the full CSV

Do not call `pd.read_csv(OUTPUT_CSV)` without `chunksize` on this large file.

In [28]:
if not OUTPUT_CSV.exists() or OUTPUT_CSV.stat().st_size == 0:
    raise RuntimeError("Output CSV is missing or empty; inspect prior cell output.")
print(f"CSV size: {OUTPUT_CSV.stat().st_size / (1024**3):.2f} GiB")
print("Columns:", pd.read_csv(OUTPUT_CSV, nrows=5).columns.tolist())
display(pd.read_csv(OUTPUT_CSV, nrows=5))
row_count = 0
popular_counts = {0: 0, 1: 0}
for part in pd.read_csv(OUTPUT_CSV, chunksize=100_000):
    row_count += len(part)
    counts = part["popular"].value_counts(dropna=False)
    popular_counts[0] += int(counts.get(0, 0))
    popular_counts[1] += int(counts.get(1, 0))
print("Validated row count:", f"{row_count:,}")
print("Popular target counts:", popular_counts)

CSV size: 2.19 GiB
Columns: ['book_id', 'user_reviews', 'user_rating', 'avg_rating', 'ratings_count', 'review_text', 'days_since_review', 'review_likes', 'like_share', 'popular']


,book_id,user_reviews,user_rating,avg_rating,ratings_count,review_text,days_since_review,review_likes,like_share,popular
0,18245960,218,5,4.01,6336.0,This is a special book. It started slow for ab...,96,29.0,0.145729,1
1,29983426,218,5,3.90,439.0,I decided to give up eating processed sugar fo...,299,32.0,0.492308,1
2,28114110,218,4,4.00,880.0,"Kevin Kelly, who is a Wired co-founder, lays o...",327,14.0,0.191781,1
3,28684704,218,3,4.10,13990.0,"A fun, fast paced science fiction thriller. I ...",353,22.0,0.021782,1
4,28119237,218,4,4.04,10078.0,A fascinating book about community and belongi...,407,32.0,0.113074,1


Validated row count: 1,981,129
Popular target counts: {0: 1572276, 1: 408853}


## 12. Download the CSV in Google Colab

Uncomment and run this cell to download the output to your computer.

In [29]:
from google.colab import files
files.download(str(OUTPUT_CSV))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Troubleshooting and notes

- If RAM usage is still high, set `REVIEW_CHUNK_SIZE = 1_000` and `LANGUAGE_BATCH_SIZE = 100`, restart runtime, and rerun from the start.
- The review file is scanned multiple times; this is slower but safer for RAM.
- Ensure there is enough disk space for both raw datasets, the SQLite database, FastText model, and final CSV.
- `popular` is the label used by the repository script. Do not use it as an input feature when training a model.
- The repository's original preprocessing also uses book metadata (`text_reviews_count`, `ratings_count`, `average_rating`). The final output retains the same ten columns as the uploaded script.
- For the paper's threshold, set `REPLICATION_MODE = "paper"` before running the notebook.